# Day I1 — 보행 주기와 보행의 언어: stride, stance, 8개 phase

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 시각 표에서 시간 변수 계산하기

In [ ]:
import numpy as np

rng = np.random.default_rng(1)          # seed 고정: 누가 돌려도 같은 가상 기록

# 가상 기록: 오른발(R)과 왼발(L)의 heel strike(HS)·toe off(TO) 시각(초)
stride_true = 1.06                      # 한 주기(초)
n = 6                                   # 오른발 주기 6개
hs_R = np.cumsum(np.r_[0.30, stride_true + rng.normal(0, 0.02, n)])
to_R = hs_R[:-1] + 0.62 * stride_true + rng.normal(0, 0.01, n)          # stance ≈ 62%
hs_L = hs_R[:-1] + 0.50 * stride_true + rng.normal(0, 0.01, n)          # 왼발은 반 주기 뒤
to_L = hs_R[1:] + 0.12 * stride_true + rng.normal(0, 0.01, n)           # 오른발 IC 뒤 약 12%에 왼발이 뜸

stride = np.diff(hs_R)                          # 오른발 stride time
stance = to_R - hs_R[:-1]                       # 오른발 stance time
swing = hs_R[1:] - to_R                         # 오른발 swing time
ds1 = to_L - hs_R[1:]                           # 다음 오른발 IC → 왼발 TO (initial double stance)
ds2 = to_R - hs_L                               # 왼발 IC → 오른발 TO (terminal double stance)
step_RL = hs_L - hs_R[:-1]                      # 오른발 IC → 왼발 IC (step time)

cadence = 60 / np.mean(np.r_[step_RL, hs_R[1:] - hs_L])   # steps/min (step 기준!)
stride_len = 1.40                               # 가정한 stride length (m)
speed = stride_len / np.mean(stride)            # m/s

print("stride time      : %.3f ± %.3f s" % (stride.mean(), stride.std(ddof=1)))
print("cadence          : %.1f steps/min (= 120 / stride time %.1f)" % (cadence, 120 / stride.mean()))
print("stance / swing   : %.1f%% / %.1f%%" % (100 * np.mean(stance / stride), 100 * np.mean(swing / stride)))
print("double stance    : initial %.1f%%, terminal %.1f%%" % (100 * np.mean(ds1 / stride), 100 * np.mean(ds2 / stride)))
print("speed            : %.2f m/s = %.0f m/min" % (speed, speed * 60))
print("검산 stance%% ≈ 50%% + 한 번의 double stance%%: %.1f ≈ 50 + %.1f"
      % (100 * np.mean(stance / stride), 100 * np.mean(ds2 / stride)))

### 예시 2 — Perry phase 표와 두 발의 막대 그림

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# Perry 8 phase의 %GC 구간과, 끝 경계에 해당하는 Whittle 사건
phases = [  # (이름, 시작%, 끝%, 끝 경계의 Whittle 사건, 과제)
    ("IC",    0,   2, "(Whittle은 IC를 순간으로 본다)",   "WA"),
    ("LR",    2,  12, "opposite toe off (OT)",            "WA"),
    ("MSt",  12,  31, "heel rise (HR)",                   "SLS"),
    ("TSt",  31,  50, "opposite initial contact (OI)",    "SLS"),
    ("PSw",  50,  62, "toe off (TO)",                     "SLA"),
    ("ISw",  62,  75, "feet adjacent (FA)",               "SLA"),
    ("MSw",  75,  87, "tibia vertical (TV)",              "SLA"),
    ("TSw",  87, 100, "다음 IC",                           "SLA"),
]
stride_s = 1.06
print("phase  %GC       길이(ms)  과제  끝 경계(Whittle 사건)")
for name, a, b, ev, task in phases:
    print("%-5s  %3d–%3d   %5.0f     %-4s  %s" % (name, a, b, (b - a) / 100 * stride_s * 1000, task, ev))

# 오른발과 왼발(50% 어긋남)의 stance/swing 막대
fig, ax = plt.subplots(figsize=(8, 2.4))
for row, shift in [(1, 0), (0, 50)]:          # 1 = 오른발, 0 = 왼발
    for a, b, color in [(0, 62, "tab:blue"), (62, 100, "lightgray")]:
        for k in (-100, 0, 100):            # 0–100% 창 밖으로 넘어가는 부분을 감싸서 그린다
            lo, hi = a + shift + k, b + shift + k
            lo, hi = max(lo, 0), min(hi, 100)
            if hi > lo:
                ax.barh(row, hi - lo, left=lo, color=color, edgecolor="k")
for x in (12, 50, 62):
    ax.axvline(x, color="r", ls="--", lw=0.8)
ax.set_yticks([0, 1], ["left", "right"])
ax.set_xlabel("% gait cycle of the right foot")
ax.set_title("blue = stance, gray = swing, red = events shared by both feet")   # 그림 글자는 영어로(한글 글꼴 문제 회피)
out = Path("figures/advanced/I/I1_phase_bar.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout()
fig.savefig(out, dpi=150)
print("저장:", out)